# Data Cleaning and Transformation

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.appName('dataTransformation').getOrCreate()

26/03/07 23:46:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
!hadoop fs -ls /data/olist/*.csv

-rw-r--r--   2 leynardlove hadoop    9033957 2026-03-06 09:18 /data/olist/olist_customers_dataset.csv
-rw-r--r--   2 leynardlove hadoop   61273883 2026-03-06 09:19 /data/olist/olist_geolocation_dataset.csv
-rw-r--r--   2 leynardlove hadoop   15438671 2026-03-06 09:19 /data/olist/olist_order_items_dataset.csv
-rw-r--r--   2 leynardlove hadoop    5777138 2026-03-06 09:19 /data/olist/olist_order_payments_dataset.csv
-rw-r--r--   2 leynardlove hadoop   14451670 2026-03-06 09:19 /data/olist/olist_order_reviews_dataset.csv
-rw-r--r--   2 leynardlove hadoop   17654914 2026-03-06 09:19 /data/olist/olist_orders_dataset.csv
-rw-r--r--   2 leynardlove hadoop    2379446 2026-03-06 09:19 /data/olist/olist_products_dataset.csv
-rw-r--r--   2 leynardlove hadoop     174703 2026-03-06 09:19 /data/olist/olist_sellers_dataset.csv
-rw-r--r--   2 leynardlove hadoop       2613 2026-03-06 09:19 /data/olist/product_category_name_translation.csv


In [5]:
# Reading the Data from HDFS

hdfs_path = '/data/olist'

customer_df = spark.read.csv(hdfs_path + '/olist_customers_dataset.csv', header=True, inferSchema=True)
orders_df = spark.read.csv(hdfs_path + '/olist_orders_dataset.csv', header=True, inferSchema=True)
order_items_df = spark.read.csv(hdfs_path + '/olist_order_items_dataset.csv', header=True, inferSchema=True)
payment_df = spark.read.csv(hdfs_path + '/olist_order_payments_dataset.csv', header=True, inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + '/olist_order_reviews_dataset.csv', header=True, inferSchema=True)
products_df = spark.read.csv(hdfs_path + '/olist_products_dataset.csv', header=True, inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + '/olist_sellers_dataset.csv', header=True, inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + '/olist_geolocation_dataset.csv', header=True, inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + '/product_category_name_translation.csv', header=True, inferSchema=True)

In [7]:
# Identify Missing Values 

def missing_values(df, df_name):
    print(f'Missing values in {df_name}:')
    df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]).show()

In [13]:
missing_values(customer_df, 'Customer Dataframe')
missing_values(orders_df, 'Order Dataframe')
missing_values(order_items_df, 'Order items Dataframe')
missing_values(payment_df, 'Payment Dataframe')
missing_values(reviews_df, 'Review Dataframe')
missing_values(products_df, 'Products Dataframe')
missing_values(sellers_df, 'Sellers Dataframe')
missing_values(geolocation_df, 'Geolocation Dataframe')
missing_values(category_translation_df, 'Category Translation Dataframe')

Missing values in Customer Dataframe:
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

Missing values in Order Dataframe:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+------------------

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+

Missing values in Products Dataframe:
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+---

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

Missing values in Category Translation Dataframe:
+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



# Handle Missing Values

1. Drop Missing Values (for non - critical columns)

2. Fill Missing values (for numerical columns)

3. Impute Missing Values (for continuous data)


In [14]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [16]:
# if order_id, customer_id, order_status are missing then we drop the row since it is an issue, 
# since without an order_id, customer_id and order_status in order 
# dataframe or order data is not logically correct.

orders_df_cleaned = orders_df.na.drop(subset=['order_id', 'customer_id', 'order_status'])

In [20]:
missing_values(orders_df_cleaned, "Order Dataframe")

Missing values in Order Dataframe:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|                0|                           0|                            0|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [137]:
# Fill the missing values specifically in dates, a date of '9999-12-31' 
# to make it or remove the missing value but put a wrong or 
# logically wrong values so that for computations we can understand that is wrong and will not give us null 


orders_df_cleaned = orders_df.fillna({'order_approved_at': '9999-12-31', 'order_delivered_carrier_date': '9999-12-31', 'order_delivered_customer_date': '9999-12-31'})
orders_df_cleaned.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [21]:
payment_df.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows



In [103]:
# Purposely make a null value in payment_df or payment_value to use imputer for missing values in numerical columns
payment_df_with_null = payment_df.withColumn('payment_value', F.when(F.col('payment_value') != 99.33, F.col('payment_value')).otherwise(F.lit(None)))

# Using imputer for study purpose 

from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=['payment_value'], outputCols=['payment_value_imputed']).setStrategy('median')

payment_df_cleaned = imputer.fit(payment_df_with_null).transform(payment_df_with_null)

payment_df_cleaned.show()

count_null = payment_df_cleaned.filter(F.col('payment_value_imputed').isNull()).count()
print(f'Null Counts: {count_null}')

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|               

In [41]:
# Make the Payment Type more readable and understandable 

payment_df_cleaned = payment_df_cleaned.withColumn('payment_type', F.when(F.col('payment_type')=='credit_card', 'Credit Card')
                                                   .when(F.col('payment_type')=='boleto', 'Bank Transfer')
                                                   .when(F.col('payment_type')=='debit_card)', 'Debit Card')
                                                   .otherwise('other')
                                                  )

In [43]:
payment_df_cleaned.show()
payment_df_cleaned.groupBy('payment_type').count().show()

+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1|  Credit Card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1|  Credit Card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1|  Credit Card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1|  Credit Card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1|  Credit Card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1|  Credit Card|                   2|        96.12|      

# Standardizing the format

In [30]:
def print_schema(df, df_name):
    print(f'Schema of {df_name}:')
    df.printSchema()

In [31]:
print_schema(orders_df, 'Order Dataframe')

Schema of Order Dataframe:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [66]:
print_schema(customer_df, 'Customer Dataframe')

Schema of Customer Dataframe:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [62]:
# Fix  customer_zip_code_prefix datatype to string 
customer_df_cleaned = customer_df.withColumn('customer_zip_code_prefix', F.col('customer_zip_code_prefix').cast('string'))

In [67]:
print_schema(customer_df_cleaned, 'Customer Cleaned Dataframe')

Schema of Customer Cleaned Dataframe:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



# Remove Duplicate 

In [70]:
customer_df.count()

99441

In [69]:
# Drop the row if there's a duplication in customer_id

customer_df_cleaned = customer_df_cleaned.dropDuplicates(['customer_id'])

In [71]:
customer_df_cleaned.count()

99441

# Joining the cleaned tables 

In [120]:
order_with_details = orders_df_cleaned.join(order_items_df, 'order_id', 'left')\
.join(payment_df_cleaned, 'order_id', 'left')\
.join(customer_df_cleaned, 'customer_id', 'left')

In [123]:
order_with_details.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+------+-------------+------------------+------------+--------------------+-------------+---------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------+------------------------+-

# Advance Transformation

In [109]:
order_items_df.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51| 199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [110]:
quantiles = order_items_df.approxQuantile('price', [0.01, 0.99], 0.01)
low_cutOff, high_cutOff = quantiles[0], quantiles[1]

In [113]:
# Remove outliers below the low_cutOff and greater than the high_cutOff from the price

order_items_df_cleaned = order_items_df.filter((F.col('price') >= low_cutOff) & (F.col('price') <= high_cutOff))
order_items_df_cleaned.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51| 199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [117]:
order_items_df_cleaned.groupBy('price').count().show()

+-------+-----+
|  price|count|
+-------+-----+
|   14.9|  296|
|  30.49|    8|
|   13.4|    1|
|  232.9|    2|
|   74.5|   21|
|  41.89|    4|
|  76.98|    2|
|   15.5|   14|
|  692.0|    1|
| 134.49|    5|
|   98.3|    7|
|   26.7|    5|
| 102.62|    3|
|  73.02|    3|
|  16.75|    4|
|  720.0|    2|
|1914.58|    2|
|   78.9|   74|
| 169.99|  232|
|   53.3|    4|
+-------+-----+
only showing top 20 rows



# Join seller_df and order_items_df_cleaned

In [128]:
order_item_with_seller = order_items_df_cleaned.join(sellers_df,'seller_id', 'left')
order_item_with_seller.show()

+--------------------+--------------------+-------------+--------------------+-------------------+------+-------------+----------------------+--------------------+------------+
|           seller_id|            order_id|order_item_id|          product_id|shipping_limit_date| price|freight_value|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+--------------------+-------------+--------------------+-------------------+------+-------------+----------------------+--------------------+------------+
|48436dade18ac8b2b...|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|2017-09-19 09:45:35|  58.9|        13.29|                 27277|       volta redonda|          SP|
|dd7ddc04e1b6c2c61...|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|2017-05-03 11:05:13| 239.9|        19.93|                  3471|           sao paulo|          SP|
|5b51032eddd242adc...|000229ec398224ef6...|            1|c777355d18b72b67a...|2018-01-18 14:48:30| 199.0|        17

In [132]:
# Total Revenue per Seller
total_revenue_per_seller = order_item_with_seller.groupBy('seller_id').agg(F.sum(F.col('price')).alias('seller_total_revenue'))
total_revenue_per_seller.show()

+--------------------+--------------------+
|           seller_id|seller_total_revenue|
+--------------------+--------------------+
|8e6cc767478edae94...|   6830.580000000001|
|4d600e08ecbe08258...|             4465.34|
|9b1050e85becf3ae9...|               85.14|
|cb5ff1b9715e99589...|                85.0|
|038b75b729c8a9a04...|               467.0|
|64c9a1db4e73e19aa...|               439.0|
|acadd4d36859671cb...|              2381.0|
|33ab10be054370c25...|  213.20000000000002|
|bec568278124768c4...|               219.9|
|b76dba6c951ab00dc...|  2574.7800000000016|
|33cbbec1e7e1044aa...|   730.6700000000001|
|e9b6c33b71b677376...|               119.9|
|7a67c85e85bb2ce85...|  141745.53000000078|
|3d8fa2f5b647373c8...|  3571.7299999999996|
|e5c84227854980f1d...|               72.81|
|9d213f303afae4983...|                47.7|
|a435b009cd956ea60...|              361.18|
|ca77545ca4d2dfd14...|               760.0|
|ee2fbacc2fc3794e6...|              464.97|
|c13ef0cfbe42f1907...|          

In [133]:
order_with_details = order_with_details.join(total_revenue_per_seller, 'seller_id', 'left')

In [134]:
order_with_details.show()

+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+-------------------+------+-------------+------------------+------------+--------------------+-------------+---------------------+--------------------+------------------------+--------------------+--------------+--------------------+
|           seller_id|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|shipping_limit_date| price|freight_value|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|seller_total_revenue|
+--------------------+------------------

# Creating a folder in HDFS and saving it in parquet

In [135]:
!hadoop fs -mkdir /data/olist_proc
order_with_details.write.mode('overwrite').parquet('/data/olist_proc/cleaned_data.parquet')

mkdir: `/data/olist_proc': File exists


In [136]:
!hadoop fs -ls /data/olist_proc/*

Found 5 items
-rw-r--r--   2 root hadoop          0 2026-03-08 03:49 /data/olist_proc/cleaned_data.parquet/_SUCCESS
-rw-r--r--   2 root hadoop    6330888 2026-03-08 03:49 /data/olist_proc/cleaned_data.parquet/part-00000-84c2173f-c0e7-43aa-8e87-61d83455b4b4-c000.snappy.parquet
-rw-r--r--   2 root hadoop    6323836 2026-03-08 03:49 /data/olist_proc/cleaned_data.parquet/part-00001-84c2173f-c0e7-43aa-8e87-61d83455b4b4-c000.snappy.parquet
-rw-r--r--   2 root hadoop    6331669 2026-03-08 03:49 /data/olist_proc/cleaned_data.parquet/part-00002-84c2173f-c0e7-43aa-8e87-61d83455b4b4-c000.snappy.parquet
-rw-r--r--   2 root hadoop    1616394 2026-03-08 03:49 /data/olist_proc/cleaned_data.parquet/part-00003-84c2173f-c0e7-43aa-8e87-61d83455b4b4-c000.snappy.parquet
